In [ ]:
!pip install polars pyarrow psutil matplotlib pandas numpy


Resolved 47 packages in 1ms
Checked 41 packages in 3ms


In [ ]:
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.csv as pa_csv
import pyarrow.compute as pc

import sys
import time
import psutil
import os
import gc
import numpy as np

In [5]:
file_path = "cleaned_data.csv"

df_check = pd.read_csv(file_path)
print(df_check.shape)
print(df_check.columns.tolist())
df_check.head()

(147558, 13)
['product_id', 'sku', 'title', 'author', 'publisher', 'current_price', 'compare_price', 'discount_pct', 'grams', 'tags', 'image_url', 'created_at', 'description']


,product_id,sku,title,author,publisher,current_price,compare_price,discount_pct,grams,tags,image_url,created_at,description
0,6978035122382,9780751547238,Visions,Kelley Armstrong,Sphere,17.9,62.90,71.5,355,"[2022], BXOS, F CTM, Fiction, Kelley Armstrong...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:08+08:00,"Olivia Jones is smart, capable, loyal...and th..."
1,6978035318990,9780751548174,The Bone Bed,Patricia Cornwell,Sphere,17.9,43.95,59.3,360,"[2022], BXOS, F CTM, Fiction, Paperback, Patri...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:08+08:00,A woman has vanished while digging a dinosaur ...
2,6978035417294,9781447205180,Jasmine Skies,Sita Brahmachari,MacMillan Children's Books,12.9,32.95,60.8,340,"[2022], BXOS, F YA, Fiction, Paperback, RM 10 ...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:09+08:00,Mira Levenson is bursting with excitement as s...
3,6978034794702,9780099461050,The Old Devils,Kingsley Amis,Vintage Publishing,17.9,59.94,70.1,310,"[2022], BXOS, F LIT, Fiction, Kingsley Amis, P...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:07+08:00,"Malcolm, Peter and Charlie and their Soave-sod..."
4,6978035515598,9781473223288,PHILIP K. DICK'S ELECTRIC DREAMS: VOLUME 1,Philip K. Dick,Gollancz,17.9,53.94,66.8,230,"[2022], BXOS, F SFF, Fiction, Paperback, Phili...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:09+08:00,"Based on the stories contained in this volume,..."


In [6]:
def flush_memory():
    gc.collect()
    time.sleep(1.0)


def result_size_mb(obj):
    """Return the in-memory footprint of a result object in MB using each library's native API."""
    try:
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            return obj.memory_usage(deep=True).sum() / (1024 ** 2)
        elif isinstance(obj, (pl.DataFrame, pl.Series)):
            return obj.estimated_size() / (1024 ** 2)
        elif isinstance(obj, (pa.Table, pa.ChunkedArray, pa.Array)):
            return obj.nbytes / (1024 ** 2)
        else:
            return sys.getsizeof(obj) / (1024 ** 2)
    except Exception:
        return 0.0


def benchmark(operation_name, library_name, func, total_rows, runs=3):
    """
    Run func `runs` times with memory flushing between runs.
    Returns a list of dicts: one row per run (1..runs) plus one
    trailing 'average' row. Memory tracked via result-object size (library native API).
    CPU is captured both before (initial) and after (final) the operation.
    """
    records = []
    exec_times, cpu_initials, cpu_finals, mems = [], [], [], []

    for run_idx in range(1, runs + 1):
        flush_memory()

        # Clean pre-operation CPU reading (after the 1-second sleep in flush_memory)
        cpu_initial = psutil.cpu_percent(interval=None)

        start  = time.perf_counter()
        result = func()
        if hasattr(result, "compute"):
            result = result.compute()
        end = time.perf_counter()

        cpu_final = psutil.cpu_percent(interval=1)  # post-operation reading
        t   = end - start
        mem = result_size_mb(result)
        tput = total_rows / t if t > 0 else 0

        records.append({
            "operation": operation_name,
            "library": library_name,
            "run": run_idx,
            "execution_time_sec": round(t, 6),
            "cpu_initial_percent": round(cpu_initial, 2),
            "cpu_final_percent": round(cpu_final, 2),
            "memory_used_mb": round(mem, 3),
            "throughput_rows_per_sec": round(tput, 0),
        })
        exec_times.append(t)
        cpu_initials.append(cpu_initial)
        cpu_finals.append(cpu_final)
        mems.append(mem)

        del result
        flush_memory()

    avg_t = float(np.mean(exec_times))
    records.append({
        "operation": operation_name,
        "library": library_name,
        "run": "average",
        "execution_time_sec": round(avg_t, 6),
        "cpu_initial_percent": round(float(np.mean(cpu_initials)), 2),
        "cpu_final_percent": round(float(np.mean(cpu_finals)), 2),
        "memory_used_mb": round(float(np.mean(mems)), 3),
        "throughput_rows_per_sec": round(total_rows / avg_t if avg_t > 0 else 0, 0),
    })
    return records

In [7]:
total_rows = len(pd.read_csv(file_path))
print("Total rows:", total_rows)

Total rows: 147558


## Pandas Benchmarks

In [8]:
results = []

# Load data into memory (not benchmarked)
pdf = pd.read_csv(file_path)
flush_memory()

# 1. Filter high-discount books
results.extend(benchmark(
    "Filter High Discount Books", "Pandas",
    lambda: pdf[(pdf["discount_pct"] > 70) & (pdf["current_price"] < 20)],
    total_rows
))

# 2. GroupBy publisher aggregation
results.extend(benchmark(
    "GroupBy Publisher Aggregation", "Pandas",
    lambda: pdf.groupby("publisher").agg({
        "current_price": "mean",
        "discount_pct": "mean",
        "title": "count"
    }),
    total_rows
))

# 3. String extract price tier
results.extend(benchmark(
    "String Extract Price Tier", "Pandas",
    lambda: pdf["tags"].astype(str).str.extract(r'(RM[\d\s\-\.]+)'),
    total_rows
))

# 4. Derived column calculation
results.extend(benchmark(
    "Calculate Savings", "Pandas",
    lambda: pdf.assign(
        savings_rm=pdf["compare_price"] - pdf["current_price"],
        savings_pct_check=(
            (pdf["compare_price"] - pdf["current_price"]) / pdf["compare_price"] * 100
        ).round(1)
    ),
    total_rows
))

# 5. Sort by discount and price
results.extend(benchmark(
    "Sort by Discount and Price", "Pandas",
    lambda: pdf.sort_values(["discount_pct", "current_price"], ascending=[False, True]),
    total_rows
))

# 6. Top authors by average discount
results.extend(benchmark(
    "Top Authors by Average Discount", "Pandas",
    lambda: pdf.groupby("author")["discount_pct"].mean().sort_values(ascending=False).head(20),
    total_rows
))

pd.DataFrame([r for r in results if r["library"] == "Pandas"])

,operation,library,run,execution_time_sec,cpu_initial_percent,cpu_final_percent,memory_used_mb,throughput_rows_per_sec
0,Filter High Discount Books,Pandas,1,0.035973,24.10,18.10,67.279,4101921.0
1,Filter High Discount Books,Pandas,2,0.024698,20.00,11.40,67.279,5974589.0
2,Filter High Discount Books,Pandas,3,0.023769,16.40,10.80,67.279,6207976.0
3,Filter High Discount Books,Pandas,average,0.028147,20.17,13.43,67.279,5242493.0
4,GroupBy Publisher Aggregation,Pandas,1,0.020558,15.70,10.10,0.459,7177748.0
5,GroupBy Publisher Aggregation,Pandas,2,0.015170,7.40,10.10,0.459,9726833.0
6,GroupBy Publisher Aggregation,Pandas,3,0.015126,7.50,7.60,0.459,9755514.0
7,GroupBy Publisher Aggregation,Pandas,average,0.016951,10.20,9.27,0.459,8704888.0
8,String Extract Price Tier,Pandas,1,0.082518,5.70,14.10,1.408,1788194.0
9,String Extract Price Tier,Pandas,2,0.074464,11.60,6.60,1.408,1981612.0


## Polars Benchmarks

In [9]:
# Load data into memory (not benchmarked)
pl_df = pl.read_csv(file_path, schema_overrides={"sku": pl.String})
flush_memory()

# 1. Filter high-discount books
results.extend(benchmark(
    "Filter High Discount Books", "Polars",
    lambda: pl_df.filter(
        (pl.col("discount_pct") > 70) & (pl.col("current_price") < 20)
    ),
    total_rows
))

# 2. GroupBy publisher aggregation
results.extend(benchmark(
    "GroupBy Publisher Aggregation", "Polars",
    lambda: pl_df.group_by("publisher").agg([
        pl.col("current_price").mean(),
        pl.col("discount_pct").mean(),
        pl.col("title").count(),
    ]),
    total_rows
))

# 3. String extract price tier
results.extend(benchmark(
    "String Extract Price Tier", "Polars",
    lambda: pl_df.select(
        pl.col("tags").str.extract(r'(RM[\d\s\-\.]+)', group_index=1)
    ),
    total_rows
))

# 4. Derived column calculation
results.extend(benchmark(
    "Calculate Savings", "Polars",
    lambda: pl_df.with_columns([
        (pl.col("compare_price") - pl.col("current_price")).alias("savings_rm"),
        ((pl.col("compare_price") - pl.col("current_price"))
         / pl.col("compare_price") * 100).round(1).alias("savings_pct_check"),
    ]),
    total_rows
))

# 5. Sort by discount and price
results.extend(benchmark(
    "Sort by Discount and Price", "Polars",
    lambda: pl_df.sort(
        by=["discount_pct", "current_price"], descending=[True, False]
    ),
    total_rows
))

# 6. Top authors by average discount
results.extend(benchmark(
    "Top Authors by Average Discount", "Polars",
    lambda: pl_df.group_by("author").agg(
        pl.col("discount_pct").mean().alias("discount_pct_mean")
    ).sort("discount_pct_mean", descending=True).head(20),
    total_rows
))

print("Polars benchmarks complete.")

Polars benchmarks complete.


## PyArrow Benchmarks

In [10]:
# Load data into memory (not benchmarked)
pa_table = pa_csv.read_csv(file_path)
flush_memory()

# 1. Filter high-discount books
results.extend(benchmark(
    "Filter High Discount Books", "PyArrow",
    lambda: pa_table.filter(
        pc.and_(
            pc.greater(pa_table.column("discount_pct"), 70),
            pc.less(pa_table.column("current_price"), 20)
        )
    ),
    total_rows
))

# 2. GroupBy publisher aggregation
results.extend(benchmark(
    "GroupBy Publisher Aggregation", "PyArrow",
    lambda: pa_table.group_by("publisher").aggregate([
        ("current_price", "mean"),
        ("discount_pct", "mean"),
        ("title", "count"),
    ]),
    total_rows
))

# 3. String extract price tier (named capture group required by PyArrow)
results.extend(benchmark(
    "String Extract Price Tier", "PyArrow",
    lambda: pc.extract_regex(
        pc.cast(pa_table.column("tags"), pa.string()),
        pattern=r'(?P<price_tier>RM[\d\s\-\.]+)'
    ),
    total_rows
))

# 4. Derived column calculation
def _pyarrow_savings():
    savings_rm = pc.subtract(pa_table.column("compare_price"), pa_table.column("current_price"))
    savings_pct = pc.round(
        pc.multiply(
            pc.divide(savings_rm, pa_table.column("compare_price")),
            100
        ),
        1
    )
    return (
        pa_table
        .append_column("savings_rm", savings_rm)
        .append_column("savings_pct_check", savings_pct)
    )

results.extend(benchmark("Calculate Savings", "PyArrow", _pyarrow_savings, total_rows))

# 5. Sort by discount and price
results.extend(benchmark(
    "Sort by Discount and Price", "PyArrow",
    lambda: pa_table.take(
        pc.sort_indices(
            pa_table,
            sort_keys=[("discount_pct", "descending"), ("current_price", "ascending")]
        )
    ),
    total_rows
))

# 6. Top authors by average discount
def _pyarrow_top_authors():
    grp = pa_table.group_by("author").aggregate([("discount_pct", "mean")])
    idx = pc.sort_indices(grp, sort_keys=[("discount_pct_mean", "descending")])
    return grp.take(idx[:20])

results.extend(benchmark("Top Authors by Average Discount", "PyArrow", _pyarrow_top_authors, total_rows))

print("PyArrow benchmarks complete.")

PyArrow benchmarks complete.


## Results Summary

In [11]:
results_df = pd.DataFrame(results)

pandas_only = results_df[results_df["library"] == "Pandas"]
pandas_only.to_csv("performance_before.csv", index=False)

results_df.to_csv("performance_after.csv", index=False)

print("Saved performance_before.csv and performance_after.csv")
results_df

Saved performance_before.csv and performance_after.csv


,operation,library,run,execution_time_sec,cpu_initial_percent,cpu_final_percent,memory_used_mb,throughput_rows_per_sec
0,Filter High Discount Books,Pandas,1,0.035973,24.10,18.10,67.279,4101921.0
1,Filter High Discount Books,Pandas,2,0.024698,20.00,11.40,67.279,5974589.0
2,Filter High Discount Books,Pandas,3,0.023769,16.40,10.80,67.279,6207976.0
3,Filter High Discount Books,Pandas,average,0.028147,20.17,13.43,67.279,5242493.0
4,GroupBy Publisher Aggregation,Pandas,1,0.020558,15.70,10.10,0.459,7177748.0
...,...,...,...,...,...,...,...,...
67,Sort by Discount and Price,PyArrow,average,0.182885,10.27,17.03,160.883,806834.0
68,Top Authors by Average Discount,PyArrow,1,0.040331,7.80,3.90,0.001,3658674.0
69,Top Authors by Average Discount,PyArrow,2,0.016219,14.40,12.90,0.001,9097904.0
70,Top Authors by Average Discount,PyArrow,3,0.019905,22.90,8.40,0.001,7413261.0


## Next Steps

Run `evaluation_charts.ipynb` to generate the charts.